# 01 — EDA and Baseline

This notebook builds a first clean baseline for the Kaggle **House Prices — Advanced Regression Techniques** competition.

The goal is not to maximize leaderboard score immediately. The goal is to create a readable, reproducible workflow: load data, inspect it, preprocess features, validate baseline models, and generate a correct submission file.

## 1. Introduction

Kaggle evaluates this competition with RMSE on the logarithm of house prices. Therefore, the reusable pipeline trains on `log1p(SalePrice)` and converts final predictions back with `expm1`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make the project package importable when this notebook is run from the notebooks folder.
PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path("projects/kaggle-house-prices").resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.config import SUBMISSION_PATH
from src.data import load_train_test
from src.features import prepare_features
from src.model import fit_best_model, save_model, train_baseline_models

## 2. Loading the data

Place `train.csv`, `test.csv`, and `sample_submission.csv` in `projects/kaggle-house-prices/data/raw/` before running this notebook.

In [ ]:
train_df, test_df = load_train_test()
train_df.head()

## 3. Train/test dimensions

A quick shape check confirms the number of rows and columns available for training and prediction.

In [ ]:
print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")

## 4. Quick SalePrice analysis

The target is right-skewed, which is why the log transformation is useful for this competition.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_df["SalePrice"].hist(bins=40, ax=axes[0])
axes[0].set_title("SalePrice distribution")
axes[0].set_xlabel("SalePrice")

np.log1p(train_df["SalePrice"]).hist(bins=40, ax=axes[1])
axes[1].set_title("log1p(SalePrice) distribution")
axes[1].set_xlabel("log1p(SalePrice)")
plt.tight_layout()

## 5. Main missing values

Missing values are common in the Ames dataset. Some indicate absence of a feature, while others are true missing measurements. The first baseline uses generic imputation; later iterations should use the data dictionary.

In [ ]:
missing = train_df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing.head(20)

## 6. Numeric and categorical variables

The baseline separates columns by dtype: numeric columns get median imputation, categorical columns get a `Missing` level.

In [ ]:
numeric_cols = train_df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = train_df.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print("Example categorical columns:", categorical_cols[:10])

## 7. Top correlations with SalePrice

Correlation is only a linear signal, but it quickly highlights important numeric variables such as overall quality and living area.

In [ ]:
corr = train_df.select_dtypes(include=["number"]).corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)
corr.head(15)

In [ ]:
corr.head(15).drop("SalePrice").plot(kind="bar", figsize=(10, 4), title="Top numeric correlations with SalePrice")
plt.ylabel("Correlation")
plt.tight_layout()

## 8. Preprocessing

The reusable preprocessing function returns aligned matrices and the log-transformed target.

In [ ]:
X_train, X_test, y = prepare_features(train_df, test_df)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Target shape:  {y.shape}")
assert list(X_train.columns) == list(X_test.columns)

## 9. Baseline Ridge

Ridge regression is a strong baseline for one-hot encoded tabular data because regularization helps control overfitting.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from src.model import rmse_cv

ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10.0, random_state=42)),
])

ridge_score = rmse_cv(ridge, X_train, y, cv=5)
print(f"Ridge CV RMSE on log target: {ridge_score:.5f}")

## 10. Baseline RandomForest

RandomForest is included as a simple nonlinear comparison. It is slower than Ridge but still reasonable for this dataset.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

rf_score = rmse_cv(rf, X_train, y, cv=5)
print(f"RandomForest CV RMSE on log target: {rf_score:.5f}")

## 11. Cross-validation RMSE comparison

The project script uses the same comparison and selects the lowest RMSE model.

In [ ]:
scores = train_baseline_models(X_train, y)
scores

## 12. Generate `submission.csv`

The predictions are produced on the log scale and converted back to prices with `expm1`. The output file has the Kaggle-required columns `Id` and `SalePrice`.

In [ ]:
best_model = fit_best_model(X_train, y)
predictions = np.expm1(best_model.predict(X_test))
predictions = np.maximum(predictions, 0)

submission = pd.DataFrame({
    "Id": test_df["Id"].values,
    "SalePrice": predictions,
})

SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(SUBMISSION_PATH, index=False)
submission.head()

In [ ]:
print(f"Submission saved to: {SUBMISSION_PATH}")
print(submission.columns.tolist())
print(submission.shape)

## 13. Model limitations

- Missing values are imputed generically instead of using domain-specific meaning.
- Skewed numeric features are not transformed yet.
- Feature engineering is minimal.
- Hyperparameters are not tuned.
- No interpretability analysis is included in this first version.

## 14. Next improvements

- Add total surface features.
- Use the Kaggle data dictionary to handle meaningful missing values.
- Log-transform highly skewed numeric features.
- Tune regularized linear models: Ridge, Lasso, ElasticNet.
- Try simple stacking.
- Add permutation importance or SHAP analysis.